# Supplemental RT Attention Overlay

Renders the Regular Transformer (RT) attention overlay with HKL annotations for **`fig:rt_attention`** in the supplement. Source script: `ai-diffraction/Code/Interpretability/interpret_reg_transformer.py`.

Two-step pipeline (mirrors the source):
1. **Spectrum + HKL generation** — `ai-diffraction/Code/Interpretability/gen_spectrum.py` runs in the `cctbx` env (no PyTorch) and writes `sample_<idx>.npz` + `sample_<idx>.json` per index into `gen_output/`. This step is *not* re-run from the notebook because it depends on the `cctbx` environment and the local `intrep_crystal_xrd.db`. The two example outputs (`sample_16`, `sample_26`) are already generated — point `GEN_DIR` below at that directory.
2. **Inference + plot** — this notebook loads the RT checkpoint with `use_flash_attn=False` (so attention weights are materialised), projects them to a 1-D profile, and overlays them on the XRD pattern with HKL labels.

Checkpoint: `hwixtnv7` (RT-RRUFF; depth=6, embed_dim=256, num_heads=4, ff_dim=128, mlp_units=256, RoPE; spec_length=8501; 10–80°). The same checkpoint backs the RT-RRUFF column of `tab:rt_distribution`.

## Environment

```bash
conda activate paper-ai-diffraction-train-eval
pip install -e .
```

Required: `torch`, `numpy`, `matplotlib`. `flash-attn` is **not** required — eager attention is forced for interpretability.

In [ ]:
import os, sys, json
from pathlib import Path
from collections import OrderedDict
import numpy as np
import torch
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')
sys.path.insert(0, str(REPO_ROOT / 'src'))

# ── Configuration (must match the checkpoint architecture) ─────────────────
CHECKPOINT  = str(REPO_ROOT / 'external' / 'checkpoints' / 'xrd_model_hwixtnv7.pth')
GEN_DIR     = str(Path.home() / 'ai-diffraction' / 'Code' / 'Interpretability' / 'gen_output')
SAVE_DIR    = str(REPO_ROOT / 'notebooks' / 'provenance' / 'rt_attention_plots')

SPEC_LENGTH = 8501
NUM_CLASSES = 99
EMBED_DIM   = 256
FF_DIM      = 128
DEPTH       = 6
NUM_HEADS   = 4
MLP_UNITS   = 256

SAMPLE_INDICES = [16, 26]  # generated in gen_output/
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
from paper_ai_diffraction.core.rt_model import transformer_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# use_flash_attn=False: forces MultiHeadAttentionWithRoPE, which exposes
# `last_attn_weights`. Flash Attention never materialises the attention matrix.
model = transformer_model(
    spec_length=SPEC_LENGTH,
    num_output=NUM_CLASSES,
    embed_dim=EMBED_DIM,
    ff_dim=FF_DIM,
    depth=DEPTH,
    num_heads=NUM_HEADS,
    mlp_units=MLP_UNITS,
    dropout=0.0,
    pos_encoding='rope',
    use_flash_attn=False,
)
ckpt = torch.load(CHECKPOINT, map_location=device)
state_dict = ckpt.get('model', ckpt)
clean = OrderedDict((k.replace('module.', ''), v) for k, v in state_dict.items())
model.load_state_dict(clean, strict=False)
model = model.to(device).eval()
print(f'Loaded checkpoint: {Path(CHECKPOINT).name}')

In [ ]:
def load_generated(gen_dir, sample_index):
    npz  = np.load(os.path.join(gen_dir, f'sample_{sample_index}.npz'))
    with open(os.path.join(gen_dir, f'sample_{sample_index}.json')) as f:
        meta = json.load(f)
    return npz['twotheta'], npz['intensity'], npz['twotheta_peaks'], meta

def get_attention(model, spectrum_np):
    x = torch.tensor(spectrum_np, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
    # last block, eager attention
    attn = model.transformer_blocks[-1].attention.last_attn_weights.squeeze(0).cpu()
    return logits.squeeze(0).cpu(), attn

def project_attention(attn_weights):
    """(num_heads, seq_len, seq_len) -> 1-D profile via mean-of-heads then column-mean.
    No CLS token in RT, so we use the column mean (how strongly each position is attended to).
    """
    avg = attn_weights.mean(0)        # (T, T)
    profile = avg.mean(0).numpy()     # (T,) — column mean
    profile /= profile.sum()
    return profile

In [ ]:
def plot_pred_vs_actual(logits, extinction_group, sample_index, save_dir, num_classes=NUM_CLASSES):
    probs = torch.softmax(logits, dim=-1)
    top5_probs, top5_preds = torch.topk(probs, 5)

    actual_idx = extinction_group - 1  # logits are 0-indexed
    pred = top5_preds[0].item()
    correct = pred == actual_idx

    class_names = [str(i) for i in range(1, num_classes + 1)]
    pred_name, actual_name = class_names[pred], class_names[actual_idx]
    labels  = [class_names[p.item()] for p in top5_preds]
    colors  = ['green' if top5_preds[i].item() == actual_idx else 'steelblue' for i in range(5)]

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(labels[::-1], top5_probs.numpy()[::-1], color=colors[::-1])
    ax.set_xlabel('Probability'); ax.set_xlim(0, 1)
    ax.set_title(
        f'Sample {sample_index} — Actual: {actual_name} | Pred: {pred_name} '
        + ('OK' if correct else 'X'),
        color='green' if correct else 'red',
    )
    for i, (prob, label) in enumerate(zip(top5_probs.numpy()[::-1], labels[::-1])):
        ax.text(prob + 0.01, i, f'{prob:.3f}', va='center', fontsize=9)
    fig.tight_layout()
    path = os.path.join(save_dir, f'pred_vs_actual_{sample_index}.png')
    fig.savefig(path, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  Saved: {path}')

def plot_attention(twotheta, intensity, twotheta_peaks, hkls, attn_profile,
                   extinction_group, crystal_system, pred_class, sample_index,
                   save_dir, n_labels=40, num_bins=50, min_x_sep=0.8, y_step=0.06):
    peak_max = intensity.max()
    y_max = peak_max * (1.20 if n_labels > 0 else 1.10)

    fig, ax = plt.subplots(figsize=(14, 5))
    attn_norm = (attn_profile - attn_profile.min()) / (attn_profile.max() - attn_profile.min() + 1e-8)
    bin_edges = np.linspace(twotheta[0], twotheta[-1], num_bins + 1)
    bin_indices = np.digitize(twotheta, bin_edges) - 1
    for b in range(num_bins):
        mask = bin_indices == b
        if np.any(mask):
            ax.fill_between(twotheta[mask], 0, y_max,
                            color=(1.0, 0.0, 0.0, float(attn_norm[mask].mean())),
                            step='mid')
    ax.plot(twotheta, intensity, 'k-', linewidth=0.8, zorder=3)

    if len(twotheta_peaks) > 0:
        peak_intensities = np.interp(twotheta_peaks, twotheta, intensity)
        top_idx = np.arange(len(hkls)) if n_labels == 0 else np.argsort(peak_intensities)[-n_labels:]
        top_idx = np.sort(top_idx)
        placed = []
        for i in top_idx:
            h, k, l = hkls[i]
            x = twotheta_peaks[i]
            y = peak_intensities[i] + 0.02 * peak_max
            for px, py in placed:
                if abs(x - px) < min_x_sep:
                    y = max(y, py + y_step * peak_max)
            placed.append((x, y))
            ax.annotate(f'({h}{k}{l})',
                        xy=(x, peak_intensities[i]), xytext=(x, y),
                        xycoords='data', textcoords='data',
                        ha='center', va='bottom', fontsize=7, rotation=45,
                        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

    ax.set_xlabel('2θ (degrees)'); ax.set_ylabel('Intensity')
    ax.set_xlim(twotheta[0], twotheta[-1]); ax.set_ylim(0, y_max)
    correct = (pred_class + 1) == extinction_group
    ax.set_title(
        f'Index {sample_index} | {crystal_system} | '
        f'Extinction group: {extinction_group} | Pred: {pred_class + 1} '
        + ('OK' if correct else 'X'),
        color='green' if correct else 'red',
    )
    path = os.path.join(save_dir, f'attn_{sample_index}_eg{extinction_group}.png')
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'  Saved: {path}')

In [ ]:
for idx in SAMPLE_INDICES:
    print(f'\n── Sample {idx} ──')
    twotheta, intensity, twotheta_peaks, meta = load_generated(GEN_DIR, idx)
    hkls = meta['hkls']
    eg   = meta['extinction_group']
    cs   = meta['crystal_system']
    print(f"  SG {meta['spacegroup_number']} | {cs} | extinction group {eg}")

    logits, attn = get_attention(model, intensity)
    pred = int(logits.argmax().item())
    print(f'  Predicted: {pred + 1} | True: {eg}')

    profile = project_attention(attn)
    plot_pred_vs_actual(logits, eg, idx, SAVE_DIR)
    plot_attention(twotheta, intensity, twotheta_peaks, hkls, profile, eg, cs, pred, idx, SAVE_DIR, n_labels=40)

print('\nDone.')